###Build Driver Standings

**Sources**\
    1. fact_session_results\
    2. dim_drivers

**Output Columns**\
    1. season\
    2. driver_id\
    3. driver_name\
    4. nationality\
    5. race starts\
    6. total points\
    7. no of wins\
    8. no of podiums\
    9. standing position

In [0]:
%sql
select * from formula1.gold.fact_session_results limit 5

In [0]:
%sql
ALTER TABLE formula1.gold.fact_session_results SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name'); ALTER TABLE formula1.gold.fact_session_results RENAME COLUMN is_podimun TO is_podium;

In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.vw_driver_standings
AS
WITH driver_session_summary
AS
    (
        SELECT r.season,
            d.driver_id,
            d.driver_name,
            d.nationality,
            COUNT(*) AS race_starts,
            SUM(r.points) AS total_points,
            count_if(r.is_win) AS wins,
            count_if(r.is_podium) AS podiums
        FROM formula1.gold.fact_session_results r
        JOIN formula1.gold.dim_drivers d
            ON r.driver_id = d.driver_id
        GROUP BY r.season,
            d.driver_id,
            d.driver_name,
            d.nationality  
            ) 
    SELECT  season,
            driver_id,
            driver_name,
            nationality,
            RANK() OVER(PARTITION BY season ORDER BY total_points DESC, wins DESC) AS standings,
            race_starts,
            total_points,
            wins,
            podiums
        FROM driver_session_summary

In [0]:
%sql
SELECT *
    FROM formula1.gold.vw_driver_standings